# Part 6 - Python Closed-Loop Control (Notebook Version)

This notebook replaces the Part 6 `.py` workflow with a step-by-step, paper-aligned analysis flow.

## What this notebook covers
1. Analyze and explain the data sources used in Part 6.
2. Visualize signals with multiple plot styles.
3. Simulate a simplified closed-loop model to build intuition.
4. Run the real `run_section43_100hz.py` script and inspect output.
5. Optionally run the full benchmark pipeline from notebook.
6. Interpret results against paper logic (NMI + adaptive admittance).

## Paper Alignment Notes

This notebook follows the same core ideas described in the paper text:
- 100 Hz closed-loop execution (`dt = 0.01 s`).
- Neuromechanical indicators (CCI and NMI).
- Adaptive behavior using NMI-informed damping/stiffness logic.
- Comparative controller view (C1/C2/C3) in a simplified simulation for understanding.

The simplified simulation here is explanatory. The authoritative benchmark is still `pipeline_arm26_full.py`.

In [ ]:
import os
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
print("Packages loaded. seaborn=", HAS_SEABORN)

In [ ]:
# Paths
ROOT = Path.cwd()
if not (ROOT / "Arm26").exists():
    # If notebook is opened from another folder, try to recover workspace root
    ROOT = Path("C:/Users/21652/Downloads/OpenSimOverView")

ARM26 = ROOT / "Arm26"
GUI_CSV = ARM26 / "OutputReference/GUI_Run_001/gui_run_dataset.csv"
TRIAL_CSV = ARM26 / "PipelineData/trials/subj_001/trial_00_nominal.csv"
LOG_100HZ = ARM26 / "OutputReference/section43_100hz_log.csv"

print("Workspace root:", ROOT)
print("GUI CSV exists:", GUI_CSV.exists())
print("Trial CSV exists:", TRIAL_CSV.exists())
print("100Hz log exists:", LOG_100HZ.exists())

## Load and Explain the Data

We use three key files:
- `gui_run_dataset.csv`: GUI -> CSV output (strict + functional EMG channels).
- `trial_00_nominal.csv`: synthetic trial data used by full pipeline stages.
- `section43_100hz_log.csv`: direct 100 Hz closed-loop reproducibility log.

In [ ]:
gui_df = pd.read_csv(GUI_CSV)
trial_df = pd.read_csv(TRIAL_CSV)
log_df = pd.read_csv(LOG_100HZ) if LOG_100HZ.exists() else None

print("GUI dataset shape:", gui_df.shape)
print("Trial dataset shape:", trial_df.shape)
if log_df is not None:
    print("100Hz log shape:", log_df.shape)

display(gui_df.head(3))
display(trial_df.head(3))
if log_df is not None:
    display(log_df.head(3))

In [ ]:
def summarize_time(df, time_col):
    t = df[time_col].to_numpy(dtype=float)
    dt = np.diff(t) if len(t) > 1 else np.array([np.nan])
    return {
        "rows": len(df),
        "t_min": float(np.nanmin(t)),
        "t_max": float(np.nanmax(t)),
        "dt_mean": float(np.nanmean(dt)),
        "dt_std": float(np.nanstd(dt)),
        "fs_est": float(1.0 / np.nanmean(dt)) if np.nanmean(dt) > 0 else np.nan,
        "nan_total": int(df.isna().sum().sum()),
    }

print("GUI summary:", summarize_time(gui_df, "time"))
print("Trial summary:", summarize_time(trial_df, "time"))
if log_df is not None:
    print("100Hz summary:", summarize_time(log_df, "time_s"))

## Plot Set 1 - Time-Series (Signals vs Time)

This shows the signal evolution over time for kinematics and EMG channels.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Kinematics
ax[0].plot(gui_df["time"], gui_df["theta_s"], label="theta_s", linewidth=1.5)
ax[0].plot(gui_df["time"], gui_df["theta_e"], label="theta_e", linewidth=1.5)
ax[0].plot(gui_df["time"], gui_df["theta_s_ref"], "--", label="theta_s_ref", alpha=0.8)
ax[0].plot(gui_df["time"], gui_df["theta_e_ref"], "--", label="theta_e_ref", alpha=0.8)
ax[0].set_ylabel("Angle (deg)")
ax[0].set_title("GUI Dataset: Joint Kinematics")
ax[0].legend(ncol=4, fontsize=9)

# Strict EMG
ax[1].plot(gui_df["time"], gui_df["emg_biceps"], label="emg_biceps")
ax[1].plot(gui_df["time"], gui_df["emg_triceps"], label="emg_triceps")
ax[1].plot(gui_df["time"], gui_df["emg_deltoid"], label="emg_deltoid")
ax[1].set_ylabel("Activation (0..1)")
ax[1].set_title("Strict EMG Mapping")
ax[1].legend()

# Functional EMG
ax[2].plot(gui_df["time"], gui_df["emg_biceps_fn"], label="emg_biceps_fn")
ax[2].plot(gui_df["time"], gui_df["emg_triceps_fn"], label="emg_triceps_fn")
ax[2].plot(gui_df["time"], gui_df["emg_deltoid_fn"], label="emg_deltoid_fn")
ax[2].set_ylabel("Activation (0..1)")
ax[2].set_xlabel("Time (s)")
ax[2].set_title("Functional EMG Mapping")
ax[2].legend()

plt.tight_layout()
plt.show()

## Plot Set 2 - Distribution Views (Histogram + Boxplot)

Different plot styles help detect bias/range/compression in EMG channels.

In [ ]:
emg_cols = [
    "emg_biceps", "emg_triceps", "emg_deltoid",
    "emg_biceps_fn", "emg_triceps_fn", "emg_deltoid_fn"
]

fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for i, col in enumerate(emg_cols):
    r, c = divmod(i, 3)
    ax[r, c].hist(gui_df[col], bins=50, alpha=0.8)
    ax[r, c].set_title(col)
    ax[r, c].set_xlabel("Value")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
gui_df[emg_cols].boxplot(rot=20)
plt.title("EMG Channel Boxplots")
plt.ylabel("Activation")
plt.tight_layout()
plt.show()

## Plot Set 3 - Correlation Heatmap + Scatter Style

Heatmaps show linear relationships; scatter-style views show trajectory structure.

In [ ]:
corr_cols = [
    "theta_s", "theta_e", "theta_s_ref", "theta_e_ref",
    "emg_biceps", "emg_triceps", "emg_deltoid",
    "emg_biceps_fn", "emg_triceps_fn", "emg_deltoid_fn"
]
corr = gui_df[corr_cols].corr()

plt.figure(figsize=(10, 8))
if HAS_SEABORN:
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
else:
    plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.xticks(range(len(corr_cols)), corr_cols, rotation=90)
    plt.yticks(range(len(corr_cols)), corr_cols)
    plt.colorbar()
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Phase-style and scatter-style plots
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(gui_df["theta_s"], gui_df["theta_e"], alpha=0.8)
plt.xlabel("theta_s (deg)")
plt.ylabel("theta_e (deg)")
plt.title("Joint Phase Portrait")

plt.subplot(1, 2, 2)
sample = gui_df.sample(min(len(gui_df), 3000), random_state=42)
plt.scatter(sample["emg_biceps"], sample["emg_triceps"], c=sample["theta_e"], s=6, alpha=0.5)
plt.xlabel("emg_biceps")
plt.ylabel("emg_triceps")
plt.title("EMG Scatter (color = theta_e)")
plt.colorbar(label="theta_e")

plt.tight_layout()
plt.show()

## Simplified Closed-Loop Simulation (C1/C2/C3)

This is an explanatory simulation inspired by the paper logic:
- C1: no assist
- C2: fixed admittance-like gains
- C3: NMI-adaptive gains

This is not a replacement for full OpenSim benchmark; it helps understanding controller behavior quickly.

In [ ]:
def compute_cci(ag, ant, eps=1e-6):
    return 2.0 * np.minimum(ag, ant) / (ag + ant + eps)

def run_simple_controller(gui_df, mode="C3"):
    t = gui_df["time"].to_numpy(dtype=float)
    dt = float(np.median(np.diff(t))) if len(t) > 1 else 0.01

    q_ref = gui_df[["theta_s_ref", "theta_e_ref"]].to_numpy(dtype=float)
    emg_b = gui_df["emg_biceps"].to_numpy(dtype=float)
    emg_t = gui_df["emg_triceps"].to_numpy(dtype=float)
    emg_d = gui_df["emg_deltoid"].to_numpy(dtype=float)

    q = np.zeros((len(t), 2), dtype=float)
    dq = np.zeros((len(t), 2), dtype=float)
    tau = np.zeros((len(t), 2), dtype=float)
    nmi_arr = np.zeros(len(t), dtype=float)

    # Fixed physical scalars for toy simulation
    I = np.array([1.2, 1.0], dtype=float)

    B_fix = np.array([2.0, 1.8])
    K_fix = np.array([18.0, 16.0])

    B_min = np.array([2.0, 1.8])
    B_max = np.array([8.0, 7.0])
    K_max = np.array([18.0, 16.0])
    K_min = np.array([6.0, 5.0])

    alpha_s, alpha_e, beta_s, beta_e = 0.4, 0.4, 0.1, 0.1
    emax_s, emax_e = 30.0, 30.0

    for i in range(1, len(t)):
        # CCI + NMI from current sample
        cci_s = compute_cci(emg_d[i], emg_t[i])
        cci_e = compute_cci(emg_b[i], emg_t[i])
        e_s = abs(q[i - 1, 0] - q_ref[i, 0]) / (emax_s + 1e-6)
        e_e = abs(q[i - 1, 1] - q_ref[i, 1]) / (emax_e + 1e-6)
        nmi = np.clip(alpha_s * cci_s + alpha_e * cci_e + beta_s * e_s + beta_e * e_e, 0.0, 1.0)
        nmi_arr[i] = nmi

        err = q_ref[i] - q[i - 1]

        if mode == "C1":
            B = np.array([0.0, 0.0])
            K = np.array([0.0, 0.0])
        elif mode == "C2":
            B = B_fix
            K = K_fix
        else:  # C3
            B = B_min + (B_max - B_min) * nmi
            K = K_max - (K_max - K_min) * nmi

        tau_i = K * err - B * dq[i - 1]
        ddq = tau_i / I

        dq[i] = dq[i - 1] + ddq * dt
        q[i] = q[i - 1] + dq[i] * dt
        tau[i] = tau_i

    mae = float(np.mean(np.abs(q - q_ref)))
    rmse = float(np.sqrt(np.mean((q - q_ref) ** 2)))
    jerk = np.diff(dq, axis=0) / dt
    jerk_int = float(np.mean(np.sum(jerk ** 2, axis=1)) * dt)

    return {
        "t": t,
        "q": q,
        "q_ref": q_ref,
        "tau": tau,
        "nmi": nmi_arr,
        "MAE": mae,
        "RMSE": rmse,
        "jerk_integrated": jerk_int,
    }

sim_c1 = run_simple_controller(gui_df, "C1")
sim_c2 = run_simple_controller(gui_df, "C2")
sim_c3 = run_simple_controller(gui_df, "C3")

summary = pd.DataFrame([
    {"controller": "C1", "MAE": sim_c1["MAE"], "RMSE": sim_c1["RMSE"], "jerk_integrated": sim_c1["jerk_integrated"]},
    {"controller": "C2", "MAE": sim_c2["MAE"], "RMSE": sim_c2["RMSE"], "jerk_integrated": sim_c2["jerk_integrated"]},
    {"controller": "C3", "MAE": sim_c3["MAE"], "RMSE": sim_c3["RMSE"], "jerk_integrated": sim_c3["jerk_integrated"]},
])
display(summary)

fig, ax = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

ax[0, 0].plot(sim_c3["t"], sim_c3["q_ref"][:, 0], "k--", label="q_sh_ref")
ax[0, 0].plot(sim_c1["t"], sim_c1["q"][:, 0], label="C1")
ax[0, 0].plot(sim_c2["t"], sim_c2["q"][:, 0], label="C2")
ax[0, 0].plot(sim_c3["t"], sim_c3["q"][:, 0], label="C3")
ax[0, 0].set_title("Shoulder Tracking")
ax[0, 0].set_ylabel("deg")
ax[0, 0].legend()

ax[0, 1].plot(sim_c3["t"], sim_c3["q_ref"][:, 1], "k--", label="q_el_ref")
ax[0, 1].plot(sim_c1["t"], sim_c1["q"][:, 1], label="C1")
ax[0, 1].plot(sim_c2["t"], sim_c2["q"][:, 1], label="C2")
ax[0, 1].plot(sim_c3["t"], sim_c3["q"][:, 1], label="C3")
ax[0, 1].set_title("Elbow Tracking")
ax[0, 1].legend()

ax[1, 0].plot(sim_c3["t"], sim_c3["tau"][:, 0], label="tau_sh C3")
ax[1, 0].plot(sim_c3["t"], sim_c3["tau"][:, 1], label="tau_el C3")
ax[1, 0].set_title("Adaptive Control Torques (C3)")
ax[1, 0].set_xlabel("Time (s)")
ax[1, 0].set_ylabel("Arbitrary units")
ax[1, 0].legend()

ax[1, 1].plot(sim_c3["t"], sim_c3["nmi"], color="firebrick")
ax[1, 1].set_title("NMI (C3 simulation)")
ax[1, 1].set_xlabel("Time (s)")
ax[1, 1].set_ylabel("NMI")

plt.tight_layout()
plt.show()

## Run the Real 100 Hz Demo (`run_section43_100hz.py`)

This cell executes the real OpenSim closed-loop script in the `biomech` conda environment and reloads the output CSV.

In [ ]:
# Full explicit command (copy/paste friendly)
cmd = [
    "conda", "run", "-n", "biomech", "python", "Arm26/run_section43_100hz.py"
]

print("Running:", " ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
print("Return code:", res.returncode)
print("--- stdout tail ---")
print("\n".join(res.stdout.splitlines()[-20:]))
if res.stderr.strip():
    print("--- stderr tail ---")
    print("\n".join(res.stderr.splitlines()[-20:]))

log_df = pd.read_csv(LOG_100HZ)
display(log_df.head(3))

fig, ax = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
ax[0].plot(log_df["time_s"], log_df["q_sh_ref_deg"], "k--", label="q_sh_ref")
ax[0].plot(log_df["time_s"], log_df["q_sh_deg"], label="q_sh")
ax[0].plot(log_df["time_s"], log_df["q_el_ref_deg"], "--", label="q_el_ref")
ax[0].plot(log_df["time_s"], log_df["q_el_deg"], label="q_el")
ax[0].set_ylabel("deg")
ax[0].set_title("Section 4.3 - Tracking at 100 Hz")
ax[0].legend(ncol=4, fontsize=9)

ax[1].plot(log_df["time_s"], log_df["tau_sh_assist_Nm"], label="tau_sh_assist")
ax[1].plot(log_df["time_s"], log_df["tau_el_assist_Nm"], label="tau_el_assist")
ax[1].set_ylabel("Nm")
ax[1].set_xlabel("Time (s)")
ax[1].set_title("Assistive Torques")
ax[1].legend()

plt.tight_layout()
plt.show()

## Optional: Run Full Benchmark Pipeline from Notebook

Set `RUN_PIPELINE = True` to execute.

Default below is quick mode + GUI CSV evaluation. You can switch to paper-style with `--runs 20` after quick validation.

In [ ]:
RUN_PIPELINE = False

# Explicit commands (copy/paste friendly)
cmd_quick = (
    "conda run -n biomech python Arm26/pipeline_arm26_full.py "
    "--quick "
    "--gui-csv Arm26/OutputReference/GUI_Run_001/gui_run_dataset.csv "
    "--data-dir Arm26/PipelineData_nb "
    "--results-dir Arm26/PipelineResults_nb"
)

cmd_paper_20 = (
    "conda run -n biomech python Arm26/pipeline_arm26_full.py "
    "--runs 20 "
    "--gui-csv Arm26/OutputReference/GUI_Run_001/gui_run_dataset.csv "
    "--data-dir Arm26/PipelineData_nb "
    "--results-dir Arm26/PipelineResults_nb"
)

results_dir = Path("Arm26/PipelineResults_nb")

if RUN_PIPELINE:
    print("Running quick pipeline...")
    print(cmd_quick)
    p = subprocess.run(cmd_quick, capture_output=True, text=True, shell=True)
    print("Return code:", p.returncode)
    print("--- stdout tail ---")
    print("\n".join(p.stdout.splitlines()[-40:]))
    if p.stderr.strip():
        print("--- stderr tail ---")
        print("\n".join(p.stderr.splitlines()[-40:]))
else:
    print("Pipeline execution skipped.")
    print("To run quick benchmark: ")
    print(cmd_quick)
    print("To run paper-style 20 runs: ")
    print(cmd_paper_20)

metrics_summary = results_dir / "metrics/summary_metrics.csv"
pairwise_stats = results_dir / "metrics/pairwise_stats.csv"

if metrics_summary.exists():
    print("Found:", metrics_summary)
    display(pd.read_csv(metrics_summary))
else:
    print("summary_metrics.csv not found yet at:", metrics_summary)

if pairwise_stats.exists():
    print("Found:", pairwise_stats)
    display(pd.read_csv(pairwise_stats).head(10))
else:
    print("pairwise_stats.csv not found yet at:", pairwise_stats)

## Interpretation Template (Paper-Oriented)

Use this checklist after running benchmark outputs:

1. **Tracking**: C3 should reduce MAE/RMSE vs C1 and usually vs C2.
2. **Neuromuscular proxy**: C3 should reduce CCI-related stress indicators in logs/features.
3. **Effort**: C3 should reduce mean RMS EMG vs C1.
4. **Stability/Smoothness**: jerk-integrated should improve or remain bounded.
5. **Stats**: inspect `pairwise_stats.csv` (normality + paired tests + corrected p-values).

If C3 is not better:
- Tune fuzzy load (`--fuzzy-tune`).
- Increase benchmark runs (`--runs 20`).
- Increase predictor training (`--pred-lstm-epochs`).
- Validate GUI CSV quality and activation mapping consistency.

# Part 6 - Professional Colab Workflow (Paper-Aligned)
This section is organized as notebook-native stages (one idea per cell), with no argparse and no main() wrapper.

## Workflow
- Setup environment and paths
- Define controller, fuzzy estimator, predictor, and decoder directly from source
- Build RMS/VAR/CCI/NMI training features
- Train Hybrid SVR-LSTM predictor
- Train QSVM intention decoder
- Tune fuzzy load estimator
- Run C1/C2/C3 closed-loop benchmark and paired statistics

In [ ]:
%pip install -q numpy pandas scipy matplotlib scikit-learn scikit-fuzzy joblib tensorflow qiskit qiskit-machine-learning

In [ ]:
import math
import random
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

ROOT = Path.cwd()
if not (ROOT / "Arm26").exists():
    ROOT = Path("/content")

GUI_CSV = ROOT / "Arm26/OutputReference/GUI_Run_001/gui_run_dataset.csv"
RESULTS_DIR = ROOT / "Arm26/PipelineResults_nb/notebook_professional"
MODEL_STORE = RESULTS_DIR / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_STORE.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("GUI_CSV:", GUI_CSV)
print("GUI exists:", GUI_CSV.exists())

## Source Cell: controller.py

In [ ]:
"""
Phase 7: Adaptive Admittance Controller.
Implements §3.7 + §4.9 of Charafeddine et al.

Admittance model (Eq.8):
    M_d * ddq + B_d(NMI) * dq + K_d(NMI) * (q - q_pred) = τ_assist

NMI-modulated bounds (Eq.9-10):
    B_d(t) = B_min + (B_max - B_min) * NMI(t)
    K_d(t) = K_max - (K_max - K_min) * NMI(t)

Higher NMI → more damping (stability) + less stiffness (compliant).
"""

import numpy as np
from dataclasses import dataclass, field


@dataclass
class AdmittanceConfig:
    # Virtual inertia (constant)
    Md_shoulder: float = 0.5     # kg·m^2
    Md_elbow:    float = 0.3

    # Damping bounds (N·m·s/rad)
    Bmin_shoulder: float = 2.0
    Bmax_shoulder: float = 20.0
    Bmin_elbow:    float = 1.0
    Bmax_elbow:    float = 10.0

    # Stiffness bounds (N·m/rad)
    Kmin_shoulder: float = 5.0
    Kmax_shoulder: float = 50.0
    Kmin_elbow:    float = 3.0
    Kmax_elbow:    float = 30.0

    # Safety torque limits (N·m)
    tau_max_shoulder: float = 30.0
    tau_max_elbow:    float = 15.0


class AdaptiveAdmittanceController:
    """
    Adaptive admittance controller with NMI-modulated B_d and K_d.

    Use modes:
        controller.compute_torque(q, dq, q_pred, nmi)
    """

    def __init__(self, cfg: AdmittanceConfig = None):
        self.cfg = cfg or AdmittanceConfig()

    def update_params(self, nmi: float):
        """Return (B_d, K_d) as 2-vectors for [shoulder, elbow]."""
        nmi = float(np.clip(nmi, 0.0, 1.0))
        B = np.array([
            self.cfg.Bmin_shoulder + (self.cfg.Bmax_shoulder - self.cfg.Bmin_shoulder) * nmi,
            self.cfg.Bmin_elbow    + (self.cfg.Bmax_elbow    - self.cfg.Bmin_elbow)    * nmi,
        ])
        K = np.array([
            self.cfg.Kmax_shoulder - (self.cfg.Kmax_shoulder - self.cfg.Kmin_shoulder) * nmi,
            self.cfg.Kmax_elbow    - (self.cfg.Kmax_elbow    - self.cfg.Kmin_elbow)    * nmi,
        ])
        return B, K

    def compute_torque(self, q, dq, q_pred, nmi: float,
                       load_kg: float = 0.0, gravity_comp_gain: float = 0.5):
        """
        Compute assistive torque (Eq.8) with NMI-modulated B_d and K_d.

        q, dq, q_pred : ndarray shape (2,)  in radians (or degrees — be consistent)
        """
        B, K = self.update_params(nmi)
        # Spring-damper assistive law
        tau = K * (q_pred - q) - B * dq

        # Load-based gravity compensation (simple model — distal forearm)
        tau[1] += gravity_comp_gain * load_kg * 9.81 * 0.3   # arm length ~0.3 m

        # Safety clamp
        tau[0] = np.clip(tau[0], -self.cfg.tau_max_shoulder, self.cfg.tau_max_shoulder)
        tau[1] = np.clip(tau[1], -self.cfg.tau_max_elbow,    self.cfg.tau_max_elbow)
        return tau


class FixedAdmittanceController:
    """Baseline (C2): constant gains, manually tuned for stability."""

    def __init__(self, B=np.array([8.0, 4.0]), K=np.array([25.0, 15.0]),
                 tau_max=np.array([30.0, 15.0])):
        self.B = B
        self.K = K
        self.tau_max = tau_max

    def compute_torque(self, q, dq, q_pred, nmi=None, load_kg=0.0):
        tau = self.K * (q_pred - q) - self.B * dq
        return np.clip(tau, -self.tau_max, self.tau_max)


class NoAssistController:
    """Baseline (C1): no torque applied."""

    def compute_torque(self, q, dq, q_pred, nmi=None, load_kg=0.0):
        return np.zeros(2)

## Source Cell: fuzzy_load.py

In [ ]:
"""
Phase 4: Fuzzy EMG-based load estimation.
Implements §3.4 + §4.6 of Charafeddine et al.

Two composite features f1, f2 (Eq.5-6) from biceps + finger-flexor proxy.
Three linguistic membership functions per input (low / medium / high) → Mamdani
rule base → centroid defuzzification → estimated load m_f(t) in [0, 2] kg.
"""

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl


def build_fuzzy_load_estimator():
    """Build and return the fuzzy control simulator."""
    f1   = ctrl.Antecedent(np.linspace(0.0, 1.0, 101), "f1")
    f2   = ctrl.Antecedent(np.linspace(0.0, 1.0, 101), "f2")
    load = ctrl.Consequent(np.linspace(0.0, 2.0, 101), "load")

    # Membership functions: low, medium, high
    for var in (f1, f2):
        var["low"]    = fuzz.trimf(var.universe, [0.00, 0.00, 0.40])
        var["medium"] = fuzz.trimf(var.universe, [0.20, 0.50, 0.80])
        var["high"]   = fuzz.trimf(var.universe, [0.60, 1.00, 1.00])

    load["light"]  = fuzz.trimf(load.universe, [0.0, 0.0, 0.8])
    load["medium"] = fuzz.trimf(load.universe, [0.4, 1.0, 1.6])
    load["heavy"]  = fuzz.trimf(load.universe, [1.2, 2.0, 2.0])

    # Compact rule base — 9 rules covering the full 3×3 space
    rules = [
        ctrl.Rule(f1["low"]    & f2["low"],    load["light"]),
        ctrl.Rule(f1["low"]    & f2["medium"], load["light"]),
        ctrl.Rule(f1["low"]    & f2["high"],   load["medium"]),

        ctrl.Rule(f1["medium"] & f2["low"],    load["light"]),
        ctrl.Rule(f1["medium"] & f2["medium"], load["medium"]),
        ctrl.Rule(f1["medium"] & f2["high"],   load["heavy"]),

        ctrl.Rule(f1["high"]   & f2["low"],    load["medium"]),
        ctrl.Rule(f1["high"]   & f2["medium"], load["heavy"]),
        ctrl.Rule(f1["high"]   & f2["high"],   load["heavy"]),
    ]
    system = ctrl.ControlSystem(rules)
    return ctrl.ControlSystemSimulation(system)


def compute_features(emg_biceps_rms, emg_biceps_var,
                     emg_fds_rms,    emg_fds_var,
                     w1: float = 0.7, w2: float = 0.7):
    """
    Composite features from Eq.5-6:
        f1 = w1 · RMS_biceps + (1 - w1) · Var(biceps)
        f2 = w2 · RMS_FDS    + (1 - w2) · Var(FDS)
    """
    f1 = w1 * emg_biceps_rms + (1.0 - w1) * emg_biceps_var
    f2 = w2 * emg_fds_rms    + (1.0 - w2) * emg_fds_var
    return float(np.clip(f1, 0.0, 1.0)), float(np.clip(f2, 0.0, 1.0))


def estimate_load(sim, f1_val: float, f2_val: float) -> float:
    """Run the fuzzy inference once and return load in kg ∈ [0, 2]."""
    try:
        sim.input["f1"] = f1_val
        sim.input["f2"] = f2_val
        sim.compute()
        return float(sim.output["load"])
    except Exception:
        return 0.5   # safe fallback


class FuzzyLoadEstimator:
    """Convenience wrapper around the fuzzy controller."""

    def __init__(self, w1: float = 0.7, w2: float = 0.7):
        self.sim = build_fuzzy_load_estimator()
        self.w1 = w1
        self.w2 = w2

    def __call__(self, emg_biceps_rms, emg_biceps_var,
                 emg_fds_rms=None, emg_fds_var=None):
        # If FDS proxy not available, use a scaled biceps signal as proxy
        if emg_fds_rms is None:
            emg_fds_rms = emg_biceps_rms * 0.8
        if emg_fds_var is None:
            emg_fds_var = emg_biceps_var * 0.8
        f1, f2 = compute_features(emg_biceps_rms, emg_biceps_var,
                                   emg_fds_rms, emg_fds_var,
                                   w1=self.w1, w2=self.w2)
        return estimate_load(self.sim, f1, f2)

## Source Cell: predictor.py

In [ ]:
"""
Phase 5: Hybrid SVR-LSTM motion predictor.
Implements §3.5 + §4.7 of Charafeddine et al.

- SVR (RBF kernel, grid-searched): static EMG → joint-angle mapping (instantaneous)
- LSTM (2 layers × 64 units, dropout 0.2, Adam 1e-3): temporal dependencies
- Hybrid: SVR output is appended to EMG features → fed as augmented input to LSTM
- Output: predicted [θ_s, θ_e] at t + Δt
"""

import numpy as np
import joblib
from pathlib import Path
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf  # noqa: E402

try:
    _THIS_FILE = Path(__file__).resolve()
    _BASE_DIR = _THIS_FILE.parent.parent
except NameError:
    _BASE_DIR = Path.cwd()
MODELS_DIR = _BASE_DIR / "data" / "trained_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
#  SVR — static mapping EMG (now) → joint angles (now)
# ============================================================

def train_svr(X_train: np.ndarray, y_train: np.ndarray, fast: bool = False):
    """
    Train a multi-output SVR with grid search over (C, gamma).

    X: (n_samples, n_features)  EMG RMS + variance features
    y: (n_samples, 2)            [θ_s, θ_e] in degrees
    """
    x_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(y_train)

    Xs = x_scaler.transform(X_train)
    ys = y_scaler.transform(y_train)

    if fast:
        base = SVR(kernel="rbf", C=10.0, gamma="scale")
        model = MultiOutputRegressor(base).fit(Xs, ys)
    else:
        param_grid = {
            "estimator__C":     [1.0, 10.0, 100.0],
            "estimator__gamma": ["scale", 0.01, 0.1],
        }
        grid = GridSearchCV(MultiOutputRegressor(SVR(kernel="rbf")),
                             param_grid, cv=3,
                             scoring="neg_mean_squared_error",
                             n_jobs=-1)
        grid.fit(Xs, ys)
        model = grid.best_estimator_

    return model, x_scaler, y_scaler


def svr_predict(model, x_scaler, y_scaler, X: np.ndarray) -> np.ndarray:
    """Return predictions in original units."""
    Xs = x_scaler.transform(X)
    ys = model.predict(Xs)
    return y_scaler.inverse_transform(ys)


# ============================================================
#  LSTM — temporal predictor
# ============================================================

def build_lstm(input_dim: int, seq_len: int = 20,
               hidden: int = 64, output_dim: int = 2,
               dropout: float = 0.2, lr: float = 1e-3) -> tf.keras.Model:
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(seq_len, input_dim)),
        tf.keras.layers.LSTM(hidden, return_sequences=True, dropout=dropout),
        tf.keras.layers.LSTM(hidden, dropout=dropout),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(output_dim),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="mse",
                  metrics=["mae"])
    return model


def make_sequences(X: np.ndarray, y: np.ndarray,
                   seq_len: int = 20, horizon: int = 5):
    """
    Convert (n_samples, n_features) into rolling sequences of length seq_len.
    Target is the value `horizon` steps in the future.
    """
    Xs, ys = [], []
    for i in range(len(X) - seq_len - horizon):
        Xs.append(X[i:i + seq_len])
        ys.append(y[i + seq_len + horizon])
    return np.array(Xs), np.array(ys)


def train_lstm(X_train: np.ndarray, y_train: np.ndarray,
               X_val:   np.ndarray, y_val:   np.ndarray,
               seq_len: int = 20, horizon: int = 5,
               epochs: int = 100, batch: int = 64, verbose: int = 1):
    """
    Train LSTM. Inputs should already be scaled. y in degrees (we scale internally).
    """
    y_scaler = StandardScaler().fit(y_train)
    y_tr = y_scaler.transform(y_train)
    y_va = y_scaler.transform(y_val)

    Xtr, ytr = make_sequences(X_train, y_tr, seq_len=seq_len, horizon=horizon)
    Xva, yva = make_sequences(X_val,   y_va, seq_len=seq_len, horizon=horizon)

    if len(Xtr) == 0:
        raise ValueError("Not enough samples to build sequences; reduce seq_len/horizon.")

    model = build_lstm(input_dim=X_train.shape[1], seq_len=seq_len)
    es = tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)

    model.fit(Xtr, ytr, validation_data=(Xva, yva),
              epochs=epochs, batch_size=batch,
              callbacks=[es], verbose=verbose)

    return model, y_scaler


class HybridPredictor:
    """SVR + LSTM combo. Stateful — keeps a rolling feature buffer."""

    def __init__(self, seq_len: int = 20, horizon: int = 5):
        self.seq_len = seq_len
        self.horizon = horizon
        self.svr = None
        self.svr_x_scaler = None
        self.svr_y_scaler = None
        self.lstm = None
        self.lstm_x_scaler = None
        self.lstm_y_scaler = None
        self.buffer = None

    def train(self, X: np.ndarray, y: np.ndarray, val_frac: float = 0.2,
              fast_svr: bool = False, lstm_epochs: int = 50, verbose: int = 1):
        n = len(X)
        n_val = int(n * val_frac)
        X_tr, X_va = X[:-n_val], X[-n_val:]
        y_tr, y_va = y[:-n_val], y[-n_val:]

        # SVR
        self.svr, self.svr_x_scaler, self.svr_y_scaler = train_svr(X_tr, y_tr, fast=fast_svr)
        svr_pred_tr = svr_predict(self.svr, self.svr_x_scaler, self.svr_y_scaler, X_tr)
        svr_pred_va = svr_predict(self.svr, self.svr_x_scaler, self.svr_y_scaler, X_va)

        # Augment LSTM input with SVR predictions
        X_tr_aug = np.hstack([X_tr, svr_pred_tr])
        X_va_aug = np.hstack([X_va, svr_pred_va])

        self.lstm_x_scaler = StandardScaler().fit(X_tr_aug)
        Xtr_s = self.lstm_x_scaler.transform(X_tr_aug)
        Xva_s = self.lstm_x_scaler.transform(X_va_aug)

        self.lstm, self.lstm_y_scaler = train_lstm(
            Xtr_s, y_tr, Xva_s, y_va,
            seq_len=self.seq_len, horizon=self.horizon,
            epochs=lstm_epochs, verbose=verbose,
        )
        self.buffer = np.zeros((self.seq_len, X_tr_aug.shape[1]))

    def predict(self, x: np.ndarray) -> np.ndarray:
        """
        Online single-step prediction.
        x: (n_features,) current EMG/joint feature vector.
        Returns predicted [θ_s, θ_e] at t + horizon * dt_window.
        """
        # 1. SVR static estimate
        svr_pred = svr_predict(self.svr, self.svr_x_scaler, self.svr_y_scaler,
                                x.reshape(1, -1))[0]
        # 2. Augment
        x_aug = np.concatenate([x, svr_pred])
        x_aug_s = self.lstm_x_scaler.transform(x_aug.reshape(1, -1))[0]
        # 3. Roll buffer
        self.buffer = np.roll(self.buffer, -1, axis=0)
        self.buffer[-1] = x_aug_s
        # 4. LSTM forward
        y_s = self.lstm.predict(self.buffer[np.newaxis, :, :], verbose=0)[0]
        return self.lstm_y_scaler.inverse_transform(y_s.reshape(1, -1))[0]

    def save(self, root: Path = MODELS_DIR):
        root = Path(root); root.mkdir(parents=True, exist_ok=True)
        joblib.dump({
            "svr": self.svr,
            "svr_x_scaler": self.svr_x_scaler,
            "svr_y_scaler": self.svr_y_scaler,
            "lstm_x_scaler": self.lstm_x_scaler,
            "lstm_y_scaler": self.lstm_y_scaler,
            "seq_len": self.seq_len,
            "horizon": self.horizon,
        }, root / "predictor_meta.joblib")
        self.lstm.save(root / "lstm.keras")

    def load(self, root: Path = MODELS_DIR):
        root = Path(root)
        meta = joblib.load(root / "predictor_meta.joblib")
        self.svr = meta["svr"]
        self.svr_x_scaler = meta["svr_x_scaler"]
        self.svr_y_scaler = meta["svr_y_scaler"]
        self.lstm_x_scaler = meta["lstm_x_scaler"]
        self.lstm_y_scaler = meta["lstm_y_scaler"]
        self.seq_len = meta["seq_len"]
        self.horizon = meta["horizon"]
        self.lstm = tf.keras.models.load_model(root / "lstm.keras")
        # buffer dimension = SVR augmented input size
        n_feat = self.lstm_x_scaler.mean_.shape[0]
        self.buffer = np.zeros((self.seq_len, n_feat))
        return self

## Source Cell: qsvm_intention.py

In [ ]:
"""
Phase 6: Quantum Support Vector Machine (QSVM) for motor intention decoding.
Implements §3.6 + §4.8 of Charafeddine et al.

- 4-qubit ZZFeatureMap (Qiskit) → quantum kernel via statevector backend
- 3 classes: 0=flexion, 1=extension, 2=hold
- Supervisory rate: 10 Hz (lower than the 100 Hz control loop)
- Safety: majority voting over recent windows + default hold on uncertainty
- Falls back to classical RBF-SVM if Qiskit is not available
"""

from collections import deque
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

try:
    _THIS_FILE = Path(__file__).resolve()
    _BASE_DIR = _THIS_FILE.parent.parent
except NameError:
    _BASE_DIR = Path.cwd()
MODELS_DIR = _BASE_DIR / "data" / "trained_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

INTENT_FLEXION   = 0
INTENT_EXTENSION = 1
INTENT_HOLD      = 2

# ---------- QSVM (Qiskit) ----------

def _build_qsvm(n_qubits: int = 4):
    """Build a QSVC with a 4-qubit ZZFeatureMap (returns None if Qiskit unavailable)."""
    try:
        from qiskit.circuit.library import ZZFeatureMap
        from qiskit_machine_learning.kernels import FidelityQuantumKernel
        from qiskit_machine_learning.algorithms import QSVC
    except Exception as e:
        print(f"[qsvm] Qiskit unavailable ({e}); using classical RBF-SVM fallback.")
        return None

    feature_map = ZZFeatureMap(feature_dimension=n_qubits, reps=2)
    kernel = FidelityQuantumKernel(feature_map=feature_map)
    return QSVC(quantum_kernel=kernel)


# ---------- Intention labels from kinematics ----------

def derive_intention_labels(theta_e: np.ndarray, dt: float, vel_threshold: float = 5.0) -> np.ndarray:
    """
    Derive ground-truth intention labels from elbow angular velocity.
        flexion   if dθ/dt > +threshold
        extension if dθ/dt < -threshold
        hold      otherwise
    """
    dtheta = np.gradient(theta_e, dt)
    labels = np.full(len(theta_e), INTENT_HOLD, dtype=int)
    labels[dtheta >  vel_threshold] = INTENT_FLEXION
    labels[dtheta < -vel_threshold] = INTENT_EXTENSION
    return labels


# ---------- Class to handle full pipeline ----------

class QSVMIntentionDecoder:
    """
    Decodes motor intention (flexion/extension/hold) from EMG features.
    Uses a 4-qubit ZZFeatureMap quantum kernel when available; falls back to
    a classical RBF-SVM otherwise (so the pipeline still runs without Qiskit).
    """

    def __init__(self, n_qubits: int = 4, vote_window: int = 5):
        self.n_qubits = n_qubits
        self.scaler = StandardScaler()
        self.pca    = PCA(n_components=n_qubits)
        self.model  = None
        self.use_qsvm = True
        self.vote_window = vote_window
        self._buffer = deque(maxlen=vote_window)

    def _build_model(self):
        m = _build_qsvm(self.n_qubits)
        if m is None:
            self.use_qsvm = False
            m = SVC(kernel="rbf", C=10.0, gamma="scale", probability=False)
        return m

    def _balance_undersample(self, X, y, rng=None):
        """Random undersampling to balance the 3 classes."""
        rng = rng or np.random.default_rng(0)
        idxs = []
        counts = np.bincount(y)
        target = counts.min()
        for c in np.unique(y):
            ids = np.flatnonzero(y == c)
            keep = rng.choice(ids, size=target, replace=False)
            idxs.append(keep)
        sel = np.concatenate(idxs)
        rng.shuffle(sel)
        return X[sel], y[sel]

    def train(self, X: np.ndarray, y: np.ndarray, max_samples: int = 800,
              random_state: int = 0):
        """
        X: (n, n_features)  EMG features
        y: (n,)              integer labels in {0,1,2}
        max_samples : cap the training set (QSVM is O(N^2 * shots))
        """
        rng = np.random.default_rng(random_state)
        # Balance
        Xb, yb = self._balance_undersample(X, y, rng=rng)
        # Cap
        if len(Xb) > max_samples:
            sel = rng.choice(len(Xb), size=max_samples, replace=False)
            Xb, yb = Xb[sel], yb[sel]

        # Scale + PCA to n_qubits dimensions
        Xs = self.scaler.fit_transform(Xb)
        Xp = self.pca.fit_transform(Xs)

        self.model = self._build_model()
        print(f"[qsvm] Training {'QSVM' if self.use_qsvm else 'classical SVM (fallback)'} "
              f"with {len(Xp)} samples × {Xp.shape[1]} features...")
        self.model.fit(Xp, yb)
        return self

    def _project(self, x: np.ndarray) -> np.ndarray:
        if x.ndim == 1:
            x = x.reshape(1, -1)
        xs = self.scaler.transform(x)
        return self.pca.transform(xs)

    def predict_raw(self, x: np.ndarray) -> int:
        return int(self.model.predict(self._project(x))[0])

    def predict_safe(self, x: np.ndarray) -> int:
        """Add to vote buffer and return majority-voted label (defaults to HOLD)."""
        raw = self.predict_raw(x)
        self._buffer.append(raw)
        if len(self._buffer) < self.vote_window:
            return INTENT_HOLD
        counts = np.bincount(list(self._buffer), minlength=3)
        if counts.max() >= int(np.ceil(self.vote_window * 0.6)):
            return int(counts.argmax())
        return INTENT_HOLD

    def save(self, path: Path = MODELS_DIR / "qsvm.joblib"):
        joblib.dump({
            "scaler": self.scaler,
            "pca": self.pca,
            "model": self.model,
            "use_qsvm": self.use_qsvm,
            "n_qubits": self.n_qubits,
            "vote_window": self.vote_window,
        }, path)

    def load(self, path: Path = MODELS_DIR / "qsvm.joblib"):
        d = joblib.load(path)
        self.scaler  = d["scaler"]
        self.pca     = d["pca"]
        self.model   = d["model"]
        self.use_qsvm = d["use_qsvm"]
        self.n_qubits = d["n_qubits"]
        self.vote_window = d["vote_window"]
        self._buffer = deque(maxlen=self.vote_window)
        return self

## Feature Engineering (RMS/VAR/CCI/NMI)

In [ ]:
PRED_FEATURE_COLS = [
    "emg_biceps_rms", "emg_triceps_rms", "emg_deltoid_rms",
    "emg_biceps_var", "emg_triceps_var", "emg_deltoid_var",
    "theta_s", "theta_e",
]
INTENT_FEATURE_COLS = [
    "emg_biceps_rms", "emg_triceps_rms", "emg_deltoid_rms",
    "emg_biceps_var", "emg_triceps_var", "emg_deltoid_var",
]
TARGET_COLS = ["theta_s_ref", "theta_e_ref"]


def compute_cci(ag: float, ant: float, eps: float = 1e-6) -> float:
    return float(2.0 * min(ag, ant) / (ag + ant + eps))


def compute_nmi(
    cci_s: float,
    cci_e: float,
    theta_s: float,
    theta_s_ref: float,
    theta_e: float,
    theta_e_ref: float,
    alpha_s: float = 0.4,
    alpha_e: float = 0.4,
    beta_s: float = 0.1,
    beta_e: float = 0.1,
    emax_s: float = 30.0,
    emax_e: float = 30.0,
) -> float:
    e_s = abs(theta_s - theta_s_ref) / (emax_s + 1e-6)
    e_e = abs(theta_e - theta_e_ref) / (emax_e + 1e-6)
    nmi = alpha_s * cci_s + alpha_e * cci_e + beta_s * e_s + beta_e * e_e
    return float(np.clip(nmi, 0.0, 1.0))


def derive_intention_from_ref(theta_e_ref: np.ndarray, dt: float, threshold_deg_s: float = 5.0) -> np.ndarray:
    vel = np.gradient(theta_e_ref, dt)
    y = np.full(theta_e_ref.shape[0], INTENT_HOLD, dtype=int)
    y[vel > threshold_deg_s] = INTENT_FLEXION
    y[vel < -threshold_deg_s] = INTENT_EXTENSION
    return y


def build_window_features(df: pd.DataFrame, fs: int = 100, window_ms: int = 200, overlap: float = 0.5) -> pd.DataFrame:
    required = [
        "time", "theta_s", "theta_e", "theta_s_ref", "theta_e_ref",
        "emg_biceps", "emg_triceps", "emg_deltoid",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in GUI CSV: {missing}")

    win = max(2, int(round(window_ms * fs / 1000.0)))
    step = max(1, int(round(win * (1.0 - overlap))))

    t = df["time"].to_numpy(dtype=float)
    eb = df["emg_biceps"].to_numpy(dtype=float)
    et = df["emg_triceps"].to_numpy(dtype=float)
    ed = df["emg_deltoid"].to_numpy(dtype=float)

    y_int = derive_intention_from_ref(df["theta_e_ref"].to_numpy(dtype=float), dt=1.0 / fs)
    rows = []

    for start in range(0, len(df) - win + 1, step):
        end = start + win
        mid = start + win // 2

        seg_b = eb[start:end]
        seg_t = et[start:end]
        seg_d = ed[start:end]

        rms_b = float(np.sqrt(np.mean(seg_b ** 2) + 1e-12))
        rms_t = float(np.sqrt(np.mean(seg_t ** 2) + 1e-12))
        rms_d = float(np.sqrt(np.mean(seg_d ** 2) + 1e-12))

        var_b = float(np.var(seg_b))
        var_t = float(np.var(seg_t))
        var_d = float(np.var(seg_d))

        theta_s = float(df["theta_s"].iloc[mid])
        theta_e = float(df["theta_e"].iloc[mid])
        theta_s_ref = float(df["theta_s_ref"].iloc[mid])
        theta_e_ref = float(df["theta_e_ref"].iloc[mid])

        cci_s = compute_cci(rms_d, rms_t)
        cci_e = compute_cci(rms_b, rms_t)
        nmi = compute_nmi(cci_s, cci_e, theta_s, theta_s_ref, theta_e, theta_e_ref)

        # Paper-friendly load proxy in [0, 2] kg range
        load_proxy = float(np.clip(2.0 * (0.6 * rms_b + 0.4 * rms_d), 0.0, 2.0))

        rows.append(
            {
                "time": float(t[mid]),
                "theta_s": theta_s,
                "theta_e": theta_e,
                "theta_s_ref": theta_s_ref,
                "theta_e_ref": theta_e_ref,
                "emg_biceps_rms": rms_b,
                "emg_triceps_rms": rms_t,
                "emg_deltoid_rms": rms_d,
                "emg_biceps_var": var_b,
                "emg_triceps_var": var_t,
                "emg_deltoid_var": var_d,
                "cci_s": cci_s,
                "cci_e": cci_e,
                "nmi": nmi,
                "intent": int(y_int[mid]),
                "load_proxy": load_proxy,
            }
        )

    return pd.DataFrame(rows)

## Build Training Table

In [ ]:
if not GUI_CSV.exists():
    raise FileNotFoundError(f"GUI CSV not found: {GUI_CSV}")

gui_df = pd.read_csv(GUI_CSV)
features_df = build_window_features(gui_df, fs=100, window_ms=200, overlap=0.5)

print("GUI rows:", len(gui_df))
print("Feature rows:", len(features_df))
print("Feature columns:", list(features_df.columns))
display(features_df.head(5))
print("Intention distribution:")
print(features_df["intent"].value_counts().sort_index())

## Train Hybrid SVR-LSTM Predictor

In [ ]:
RUN_TRAIN_PREDICTOR = True
PRED_ROOT = MODEL_STORE / "predictor"

if RUN_TRAIN_PREDICTOR:
    X_pred = features_df[PRED_FEATURE_COLS].to_numpy(dtype=float)
    Y_pred = features_df[TARGET_COLS].to_numpy(dtype=float)

    predictor = HybridPredictor(seq_len=10, horizon=3)
    predictor.train(
        X_pred,
        Y_pred,
        val_frac=0.2,
        fast_svr=True,
        lstm_epochs=12,
        verbose=0,
    )
    predictor.save(root=PRED_ROOT)
    print("Predictor trained and saved to:", PRED_ROOT)
else:
    predictor = HybridPredictor().load(root=PRED_ROOT)
    print("Predictor loaded from:", PRED_ROOT)

## Train QSVM Intention Decoder

In [ ]:
RUN_TRAIN_DECODER = True
DECODER_PATH = MODEL_STORE / "qsvm.joblib"

if RUN_TRAIN_DECODER:
    X_dec = features_df[INTENT_FEATURE_COLS].to_numpy(dtype=float)
    y_dec = features_df["intent"].to_numpy(dtype=int)

    decoder = QSVMIntentionDecoder(n_qubits=4, vote_window=5)
    decoder.train(
        X_dec,
        y_dec,
        max_samples=min(600, len(X_dec)),
        random_state=SEED,
    )
    decoder.save(path=DECODER_PATH)
    print("Decoder trained and saved to:", DECODER_PATH)
else:
    decoder = QSVMIntentionDecoder().load(path=DECODER_PATH)
    print("Decoder loaded from:", DECODER_PATH)

## Tune Fuzzy Load Estimator

In [ ]:
def tune_fuzzy(df: pd.DataFrame, grid: np.ndarray = np.linspace(0.4, 0.9, 6)):
    best = {"w1": 0.7, "w2": 0.7, "mae": float("inf")}

    target = df["load_proxy"].to_numpy(dtype=float)
    for w1 in grid:
        for w2 in grid:
            est = FuzzyLoadEstimator(w1=float(w1), w2=float(w2))
            preds = []
            for _, row in df.iterrows():
                p = est(
                    float(row["emg_biceps_rms"]),
                    float(row["emg_biceps_var"]),
                    emg_fds_rms=0.8 * float(row["emg_biceps_rms"]),
                    emg_fds_var=0.8 * float(row["emg_biceps_var"]),
                )
                preds.append(p)

            mae = float(np.mean(np.abs(np.asarray(preds) - target)))
            if mae < best["mae"]:
                best = {"w1": float(w1), "w2": float(w2), "mae": mae}

    return FuzzyLoadEstimator(w1=best["w1"], w2=best["w2"]), best


fuzzy, fuzzy_cfg = tune_fuzzy(features_df)
print("Best fuzzy config:", fuzzy_cfg)

## Closed-Loop Simulation (C1/C2/C3, Paper Logic)

In [ ]:
def run_closed_loop_sim(
    feat_df: pd.DataFrame,
    mode: str,
    predictor: HybridPredictor,
    decoder: QSVMIntentionDecoder,
    fuzzy: FuzzyLoadEstimator,
    dt: float = 0.01,
    decode_hz: float = 10.0,
):
    t = feat_df["time"].to_numpy(dtype=float)
    q_ref = feat_df[["theta_s_ref", "theta_e_ref"]].to_numpy(dtype=float)

    rms_cols = ["emg_biceps_rms", "emg_triceps_rms", "emg_deltoid_rms"]
    var_cols = ["emg_biceps_var", "emg_triceps_var", "emg_deltoid_var"]
    emg_r = feat_df[rms_cols].to_numpy(dtype=float)
    emg_v = feat_df[var_cols].to_numpy(dtype=float)

    q = np.zeros_like(q_ref, dtype=float)
    dq = np.zeros_like(q_ref, dtype=float)
    tau = np.zeros_like(q_ref, dtype=float)

    nmi_arr = np.zeros(len(t), dtype=float)
    cci_arr = np.zeros(len(t), dtype=float)
    load_hat_arr = np.zeros(len(t), dtype=float)
    intent_arr = np.full(len(t), INTENT_HOLD, dtype=int)

    I = np.array([1.2, 1.0], dtype=float)
    fixed_ctrl = FixedAdmittanceController()
    adaptive_ctrl = AdaptiveAdmittanceController()

    if predictor.buffer is not None:
        predictor.buffer = np.zeros_like(predictor.buffer)
    decoder._buffer = deque(maxlen=decoder.vote_window)

    decode_step = max(1, int(round(1.0 / (dt * decode_hz))))
    current_intent = INTENT_HOLD

    for i in range(1, len(t)):
        rms_b, rms_t, rms_d = emg_r[i]
        var_b, var_t, var_d = emg_v[i]

        cci_s = compute_cci(rms_d, rms_t)
        cci_e = compute_cci(rms_b, rms_t)
        cci_arr[i] = 0.5 * (cci_s + cci_e)

        nmi = compute_nmi(
            cci_s,
            cci_e,
            q[i - 1, 0],
            q_ref[i, 0],
            q[i - 1, 1],
            q_ref[i, 1],
        )
        nmi_arr[i] = nmi

        if mode == "C3":
            x_pred = np.array([
                rms_b, rms_t, rms_d,
                var_b, var_t, var_d,
                q[i - 1, 0], q[i - 1, 1],
            ], dtype=float)
            q_pred = predictor.predict(x_pred)

            if i % decode_step == 0:
                f_intent = np.array([rms_b, rms_t, rms_d, var_b, var_t, var_d], dtype=float)
                current_intent = decoder.predict_safe(f_intent)

            load_hat = fuzzy(
                rms_b,
                var_b,
                emg_fds_rms=0.8 * rms_b,
                emg_fds_var=0.8 * var_b,
            )
            load_hat_arr[i] = float(load_hat)
        else:
            q_pred = q_ref[i]
            load_hat = 0.0

        intent_arr[i] = current_intent

        if mode == "C1":
            tau_i = np.zeros(2, dtype=float)
        elif mode == "C2":
            tau_i = np.asarray(
                fixed_ctrl.compute_torque(
                    np.deg2rad(q[i - 1]),
                    np.deg2rad(dq[i - 1]),
                    np.deg2rad(q_ref[i]),
                    nmi=None,
                    load_kg=0.0,
                ),
                dtype=float,
            )
        elif mode == "C3":
            tau_i = np.asarray(
                adaptive_ctrl.compute_torque(
                    np.deg2rad(q[i - 1]),
                    np.deg2rad(dq[i - 1]),
                    np.deg2rad(q_pred),
                    nmi=nmi,
                    load_kg=float(load_hat),
                ),
                dtype=float,
            )
            if current_intent == INTENT_HOLD:
                tau_i *= 0.3
        else:
            raise ValueError(f"Unknown mode: {mode}")

        ddq = tau_i / I
        dq[i] = dq[i - 1] + ddq * dt
        q[i] = q[i - 1] + dq[i] * dt
        tau[i] = tau_i

    err = q - q_ref
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mean_rms_emg = float(emg_r.mean())
    mean_cci = float(cci_arr.mean())

    jerk = np.diff(np.deg2rad(q), n=3, axis=0) / (dt ** 3)
    jerk_integrated = float(np.sum(jerk ** 2)) if len(jerk) > 0 else 0.0

    metrics = {
        "MAE_deg": mae,
        "RMSE_deg": rmse,
        "mean_RMS_EMG": mean_rms_emg,
        "mean_CCI": mean_cci,
        "jerk_integrated": jerk_integrated,
    }

    log_df = pd.DataFrame(
        {
            "time": t,
            "q_sh": q[:, 0],
            "q_el": q[:, 1],
            "q_sh_ref": q_ref[:, 0],
            "q_el_ref": q_ref[:, 1],
            "tau_sh": tau[:, 0],
            "tau_el": tau[:, 1],
            "nmi": nmi_arr,
            "cci": cci_arr,
            "load_hat": load_hat_arr,
            "intent": intent_arr,
            "mode": mode,
        }
    )

    return {"metrics": metrics, "log": log_df}


def paired_stats(metric_name: str, c1: np.ndarray, c2: np.ndarray, c3: np.ndarray) -> pd.DataFrame:
    pairs = [("C1 vs C2", c1, c2), ("C1 vs C3", c1, c3), ("C2 vs C3", c2, c3)]
    rows = []
    for label, a, b in pairs:
        diff = a - b
        if len(diff) >= 3:
            _, p_norm = stats.shapiro(diff)
        else:
            p_norm = 0.0

        if p_norm > 0.05:
            test = "paired_t"
            stat, p = stats.ttest_rel(a, b)
        else:
            test = "wilcoxon"
            try:
                stat, p = stats.wilcoxon(a, b)
            except ValueError:
                stat, p = np.nan, 1.0

        rows.append(
            {
                "metric": metric_name,
                "pair": label,
                "test": test,
                "statistic": float(stat),
                "p_raw": float(p),
                "p_bonferroni": float(min(p * 3.0, 1.0)),
            }
        )
    return pd.DataFrame(rows)

## Multi-Run Benchmark + Paired Stats

In [ ]:
RUN_BENCHMARK = True
BENCHMARK_RUNS = 5

results = {"C1": [], "C2": [], "C3": []}
first_logs = {}

if RUN_BENCHMARK:
    emg_noise_cols = [
        "emg_biceps_rms", "emg_triceps_rms", "emg_deltoid_rms",
        "emg_biceps_var", "emg_triceps_var", "emg_deltoid_var",
    ]

    for run in range(BENCHMARK_RUNS):
        noisy = features_df.copy()
        for col in emg_noise_cols:
            noisy[col] = np.clip(
                noisy[col].to_numpy(dtype=float) + np.random.normal(0.0, 0.01, len(noisy)),
                0.0,
                1.0,
            )

        out_c1 = run_closed_loop_sim(noisy, "C1", predictor, decoder, fuzzy)
        out_c2 = run_closed_loop_sim(noisy, "C2", predictor, decoder, fuzzy)
        out_c3 = run_closed_loop_sim(noisy, "C3", predictor, decoder, fuzzy)

        if run == 0:
            first_logs = {"C1": out_c1["log"], "C2": out_c2["log"], "C3": out_c3["log"]}

        m1 = dict(out_c1["metrics"])
        m2 = dict(out_c2["metrics"])
        m3 = dict(out_c3["metrics"])

        m1["run"] = run
        m2["run"] = run
        m3["run"] = run

        m1["EMG_reduction_%"] = 0.0
        m1["CCI_reduction_%"] = 0.0

        m2["EMG_reduction_%"] = 100.0 * (1.0 - m2["mean_RMS_EMG"] / max(m1["mean_RMS_EMG"], 1e-9))
        m2["CCI_reduction_%"] = 100.0 * (1.0 - m2["mean_CCI"] / max(m1["mean_CCI"], 1e-9))

        m3["EMG_reduction_%"] = 100.0 * (1.0 - m3["mean_RMS_EMG"] / max(m1["mean_RMS_EMG"], 1e-9))
        m3["CCI_reduction_%"] = 100.0 * (1.0 - m3["mean_CCI"] / max(m1["mean_CCI"], 1e-9))

        results["C1"].append(m1)
        results["C2"].append(m2)
        results["C3"].append(m3)

    metrics_c1 = pd.DataFrame(results["C1"])
    metrics_c2 = pd.DataFrame(results["C2"])
    metrics_c3 = pd.DataFrame(results["C3"])

    def summarize(df: pd.DataFrame, controller: str) -> pd.DataFrame:
        s = df.agg(["mean", "std"]).T.reset_index().rename(columns={"index": "metric"})
        s.insert(0, "controller", controller)
        return s

    summary_df = pd.concat(
        [
            summarize(metrics_c1, "C1"),
            summarize(metrics_c2, "C2"),
            summarize(metrics_c3, "C3"),
        ],
        ignore_index=True,
    )

    stats_df = pd.concat(
        [
            paired_stats("MAE_deg", metrics_c1["MAE_deg"].values, metrics_c2["MAE_deg"].values, metrics_c3["MAE_deg"].values),
            paired_stats("RMSE_deg", metrics_c1["RMSE_deg"].values, metrics_c2["RMSE_deg"].values, metrics_c3["RMSE_deg"].values),
            paired_stats("mean_RMS_EMG", metrics_c1["mean_RMS_EMG"].values, metrics_c2["mean_RMS_EMG"].values, metrics_c3["mean_RMS_EMG"].values),
            paired_stats("mean_CCI", metrics_c1["mean_CCI"].values, metrics_c2["mean_CCI"].values, metrics_c3["mean_CCI"].values),
            paired_stats("jerk_integrated", metrics_c1["jerk_integrated"].values, metrics_c2["jerk_integrated"].values, metrics_c3["jerk_integrated"].values),
            paired_stats("EMG_reduction_%", metrics_c1["EMG_reduction_%"].values, metrics_c2["EMG_reduction_%"].values, metrics_c3["EMG_reduction_%"].values),
            paired_stats("CCI_reduction_%", metrics_c1["CCI_reduction_%"].values, metrics_c2["CCI_reduction_%"].values, metrics_c3["CCI_reduction_%"].values),
        ],
        ignore_index=True,
    )

    display(summary_df)
    display(stats_df.head(12))
else:
    print("Benchmark skipped (RUN_BENCHMARK=False).")

## Save Results and Plot

In [ ]:
if RUN_BENCHMARK:
    metrics_c1.to_csv(RESULTS_DIR / "metrics_C1.csv", index=False)
    metrics_c2.to_csv(RESULTS_DIR / "metrics_C2.csv", index=False)
    metrics_c3.to_csv(RESULTS_DIR / "metrics_C3.csv", index=False)
    summary_df.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)
    stats_df.to_csv(RESULTS_DIR / "pairwise_stats.csv", index=False)

    if first_logs:
        first_logs["C1"].to_csv(RESULTS_DIR / "log_C1_run01.csv", index=False)
        first_logs["C2"].to_csv(RESULTS_DIR / "log_C2_run01.csv", index=False)
        first_logs["C3"].to_csv(RESULTS_DIR / "log_C3_run01.csv", index=False)

        fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
        ax[0].plot(first_logs["C3"]["time"], first_logs["C3"]["q_sh_ref"], "k--", label="q_sh_ref")
        ax[0].plot(first_logs["C1"]["time"], first_logs["C1"]["q_sh"], label="C1")
        ax[0].plot(first_logs["C2"]["time"], first_logs["C2"]["q_sh"], label="C2")
        ax[0].plot(first_logs["C3"]["time"], first_logs["C3"]["q_sh"], label="C3")
        ax[0].set_ylabel("Shoulder (deg)")
        ax[0].legend()

        ax[1].plot(first_logs["C3"]["time"], first_logs["C3"]["nmi"], color="firebrick", label="NMI C3")
        ax[1].set_xlabel("Time (s)")
        ax[1].set_ylabel("NMI")
        ax[1].legend()
        plt.tight_layout()
        plt.show()

    print("Saved results in:", RESULTS_DIR)